In [ ]:
# Path to the ISIC 2018 Task 3 download.
# Expects the official Training / Validation / Test Input folders and
# GroundTruth csv files inside it.
DATA_DIR = r"PATH/TO/ISIC2018"


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_fscore_support
)

# ---- Basic setup -------------------------------------------------------
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---- Paths (EDIT THESE SIX to match your machine) ----------------------
# Your supervisor asked us to use ISIC's own official Training / Validation
# / Test releases instead of carving validation and test out of the
# training data ourselves. So now there are three separate CSVs and three
# separate image folders - one pair per official release.
TRAIN_CSV_PATH = DATA_DIR + r"\ISIC2018_Task3_Training_GroundTruth\ISIC2018_Task3_Training_GroundTruth.csv"
TRAIN_IMAGE_FOLDER = DATA_DIR + r"\ISIC2018_Task3_Training_Input"

VAL_CSV_PATH = DATA_DIR + r"\ISIC2018_Task3_Validation_GroundTruth\ISIC2018_Task3_Validation_GroundTruth.csv"
VAL_IMAGE_FOLDER = DATA_DIR + r"\ISIC2018_Task3_Validation_Input"

TEST_CSV_PATH = DATA_DIR + r"\ISIC2018_Task3_Test_GroundTruth\ISIC2018_Task3_Test_GroundTruth.csv"
TEST_IMAGE_FOLDER = DATA_DIR + r"\ISIC2018_Task3_Test_Input"

IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
def get_binary_label(row):
    if row['MEL'] == 1 or row['BCC'] == 1 or row['AKIEC'] == 1:
        return 'Malignant'
    return 'Benign'

In [ ]:
# One simple function that does the same 4 steps for whichever official
# release we give it: read the CSV, turn the 7 labels into Benign/Malignant,
# build the full image path, and check the image is actually on disk.
# We call this once each for Training, Validation and Test - so all three
# get treated exactly the same way, just pointed at different folders.
def load_and_prepare(csv_path, image_folder, name):
    data = pd.read_csv(csv_path)

    data['Binary_Class'] = data.apply(get_binary_label, axis=1)
    data['label'] = (data['Binary_Class'] == 'Malignant').astype(int)

    data['image_path'] = data['image'].apply(
        lambda x: os.path.normpath(os.path.join(image_folder, x + ".jpg"))
    )
    data['exists'] = data['image_path'].apply(os.path.exists)

    n_total = len(data)
    n_found = int(data['exists'].sum())
    print(f"{name}: {n_total} referenced, {n_found} found on disk, {n_total - n_found} missing")

    data = data[data['exists']].reset_index(drop=True)
    return data

train_df = load_and_prepare(TRAIN_CSV_PATH, TRAIN_IMAGE_FOLDER, "Training")
val_df = load_and_prepare(VAL_CSV_PATH, VAL_IMAGE_FOLDER, "Validation")
test_df = load_and_prepare(TEST_CSV_PATH, TEST_IMAGE_FOLDER, "Test")

print()
print("Training  :", len(train_df), "-", train_df['Binary_Class'].value_counts().to_dict())
print("Validation:", len(val_df), "-", val_df['Binary_Class'].value_counts().to_dict())
print("Testing   :", len(test_df), "-", test_df['Binary_Class'].value_counts().to_dict())

In [ ]:
# Same idea as before: tell the model to pay extra attention to the rare
# malignant cases. In PyTorch this is one number (pos_weight) instead of a
# dictionary of two weights, but it does the same job.
class_weights_array = compute_class_weight(
    class_weight='balanced', classes=np.array([0, 1]), y=train_df['label'].values
)
pos_weight_value = class_weights_array[1] / class_weights_array[0]
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
print(f"pos_weight (how much extra attention to give malignant cases): {pos_weight_value:.4f}")

In [ ]:
# A simple "recipe" for loading one image + its label at a time.
class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        image = self.transform(image)
        label = torch.tensor(row['label'], dtype=torch.float32)
        return image, label

In [ ]:
# Data augmentation for the training images. Your Terms of Reference
# specifically lists 5 techniques (section 6.3) — rotation, flipping,
# zooming, cropping, and brightness adjustment — so all 5 are included
# here, each labelled with which one it is.
basic_train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),  # zooming + cropping
    T.RandomRotation(20),                              # rotation
    T.RandomHorizontalFlip(),                          # flipping
    T.RandomVerticalFlip(),                             # flipping
    T.ColorJitter(brightness=0.15),                     # brightness adjustment
    T.ToTensor(),
])

basic_eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
])

train_ds = SkinLesionDataset(train_df, basic_train_transform)
val_ds = SkinLesionDataset(val_df, basic_eval_transform)
test_ds = SkinLesionDataset(test_df, basic_eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# One function that trains a model for a fixed number of epochs. It keeps
# track of the best validation score it's seen, and saves a copy of the
# model at that point — so even if later epochs get slightly worse, you
# still have the best version saved to disk. No early-stopping logic and
# no learning-rate scheduler here, to keep this easy to follow; it's just
# a plain loop plus "if this is the best so far, save it."
def train_model(model, train_loader, val_loader, epochs, lr, save_path=None):
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    for epoch in range(epochs):
        # ---- learn from the training images ----
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = total_loss / total
        train_acc = correct / total

        # ---- check performance on validation images (no learning here) ----
        model.eval()
        val_loss_total, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device).unsqueeze(1)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss_total += loss.item() * images.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss_total / val_total
        val_acc = val_correct / val_total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- keep the best version seen so far ----
        improved = ""
        if save_path and val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            improved = "  <- best so far, saved"

        print(f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - acc: {train_acc:.4f} "
              f"- val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}{improved}")

    return history

In [ ]:
def plot_history(history, title):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(history['train_acc'], marker='o', label='train')
    ax[0].plot(history['val_acc'], marker='s', label='val')
    ax[0].set_title(f'{title} — Accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(True)
    ax[1].plot(history['train_loss'], marker='o', label='train')
    ax[1].plot(history['val_loss'], marker='s', label='val')
    ax[1].set_title(f'{title} — Loss'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(True)
    plt.tight_layout(); plt.show()

In [ ]:
def evaluate_model(model, test_loader, model_name="model"):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy().ravel()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.numpy().ravel().tolist())

    y_prob = np.array(all_probs)
    y_true = np.array(all_labels).astype(int)
    y_pred = (y_prob >= 0.5).astype(int)

    print(f"===== {model_name} =====")
    print(classification_report(y_true, y_pred, target_names=['Benign', 'Malignant'], digits=4))

    auc = roc_auc_score(y_true, y_prob)
    print(f"ROC-AUC: {auc:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Malignant']).plot(
        cmap='Blues', ax=ax[0], colorbar=False)
    ax[0].set_title(f"{model_name} — Confusion Matrix")

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ax[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc:.3f}')
    ax[1].plot([0, 1], [0, 1], 'k--', lw=1)
    ax[1].set_xlabel('False Positive Rate'); ax[1].set_ylabel('True Positive Rate')
    ax[1].set_title(f"{model_name} — ROC Curve"); ax[1].legend(loc='lower right')
    plt.tight_layout(); plt.show()

    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=1, zero_division=0)
    return {'model': model_name, 'accuracy': (y_pred == y_true).mean(),
            'precision_malignant': p, 'recall_malignant': r,
            'f1_malignant': f1, 'roc_auc': auc}

results = []

In [ ]:
# The basic CNN, built from scratch — same design as your Keras version:
# three blocks of (look for patterns -> shrink), then decide.
class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(128, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)   # flatten
        x = self.dropout(x)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)             # raw score — sigmoid applied inside the loss / when evaluating
        return x

In [ ]:
baseline_model = BaselineCNN().to(device)
history_cnn = train_model(baseline_model, train_loader, val_loader,
                           epochs=15, lr=1e-3, save_path="best_cnn.pt")

In [ ]:
plot_history(history_cnn, "Baseline CNN")
results.append(evaluate_model(baseline_model, test_loader, "Baseline CNN"))

In [ ]:
# Pretrained models need images normalised the same way they were trained
# originally (standard ImageNet numbers) instead of a plain 0-1 scale.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

pretrained_train_transform = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),  # zooming + cropping
    T.RandomRotation(20),                              # rotation
    T.RandomHorizontalFlip(),                          # flipping
    T.RandomVerticalFlip(),                             # flipping
    T.ColorJitter(brightness=0.15),                     # brightness adjustment
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

pretrained_eval_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds_t = SkinLesionDataset(train_df, pretrained_train_transform)
val_ds_t = SkinLesionDataset(val_df, pretrained_eval_transform)
test_ds_t = SkinLesionDataset(test_df, pretrained_eval_transform)

train_loader_t = DataLoader(train_ds_t, batch_size=BATCH_SIZE, shuffle=True)
val_loader_t = DataLoader(val_ds_t, batch_size=BATCH_SIZE, shuffle=False)
test_loader_t = DataLoader(test_ds_t, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# EfficientNetB0 — Step 1: use the pretrained "expert" as-is, only train
# a brand-new decision-making layer on top of it.
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

for param in effnet.parameters():
    param.requires_grad = False   # freeze everything the expert already knows

effnet.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(effnet.classifier[1].in_features, 1)
)
effnet = effnet.to(device)

history_eff1 = train_model(effnet, train_loader_t, val_loader_t, epochs=5, lr=1e-3)

In [ ]:
# EfficientNetB0 — Step 2: gently let the last few layers adjust too,
# using a much smaller learning rate so we don't wreck what it already knows.
for param in list(effnet.parameters())[-10:]:
    param.requires_grad = True

history_eff2 = train_model(effnet, train_loader_t, val_loader_t,
                            epochs=10, lr=1e-5, save_path="best_effnet.pt")

In [ ]:
plot_history(history_eff2, "EfficientNetB0 (fine-tune)")
results.append(evaluate_model(effnet, test_loader_t, "EfficientNetB0"))

In [ ]:
# DenseNet201 — same two-step idea as EfficientNet above.
densenet = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)

for param in densenet.parameters():
    param.requires_grad = False

densenet.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(densenet.classifier.in_features, 1)
)
densenet = densenet.to(device)

history_dn1 = train_model(densenet, train_loader_t, val_loader_t, epochs=5, lr=1e-3)

In [ ]:
for param in list(densenet.parameters())[-10:]:
    param.requires_grad = True

history_dn2 = train_model(densenet, train_loader_t, val_loader_t,
                           epochs=10, lr=1e-5, save_path="best_densenet.pt")

In [ ]:
plot_history(history_dn2, "DenseNet201 (fine-tune)")
results.append(evaluate_model(densenet, test_loader_t, "DenseNet201"))

In [ ]:
comparison = pd.DataFrame(results).set_index('model').round(4)
comparison